In [1]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# 1. 크롬 드라이버 초기화 및 옵션 설정
# ==============================================================================
option = webdriver.ChromeOptions()
option.add_argument('--headless')
option.add_argument('--no-sandbox')
option.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome(options=option)
driver.implicitly_wait(10) # 암묵적 대기 설정

# 2. 지자체별 정보 정의 (핵심 부분)
# ==============================================================================
local_governments = [
    {
        "name": "전남",
        "url": "https://www.jeonnam.go.kr/J0203/boardList.do?menuId=jeonnam0203000000",
        "search_input_xpath": '//*[@id="search_board"]',
        "search_btn_xpath": '//*[@id="frm"]/fieldset/div/div/div[2]/input[2]',
        "results_xpath": '//*[@id="frm"]/div[1]/table/tbody/tr'
    },
    {
        "name": "전북",
        "url": "https://www.jeonbuk.go.kr/board/list.jeonbuk?boardId=BBS_0000129&menuCd=DOM_000000102002005000&contentsSid=1379&cpath=",
        "search_input_xpath": '//*[@id="keyword"]',
        "search_btn_xpath": '/html/body/div/section/div[2]/article/div[2]/div[1]/form/fieldset/div/input[3]',
        "results_xpath": '/html/body/div/section/div[2]/article/div[2]/div[2]/table/tbody/tr'
    },
     {
        "name": "전북타기관",
        "url": "https://www.jeonbuk.go.kr/board/list.jeonbuk?boardId=BBS_0000006&menuCd=DOM_000000102002006000&contentsSid=1380&cpath=",
        "search_input_xpath": '//*[@id="keyword"]',
        "search_btn_xpath": '/html/body/div/section/div[2]/article/div/div[1]/form/fieldset/div/input[4]',
        "results_xpath": '/html/body/div/section/div[2]/article/div/div[2]/table/tbody/tr'
    },
    {
        "name": "광주",
        "url": "https://search.gwangju.go.kr/search/front/Search.jsp",
        "search_input_xpath": '//*[@id="input_keyword"]',
        "search_btn_xpath": '//*[@id="searchForm"]/div[2]/div[2]/a',
        "results_xpath": '//*[@id="content_area"]/div/div/div[5]/div'
    },
    {
        "name": "경기",
        "url": "https://www.gg.go.kr/bbs/board.do?bsIdx=469&menuId=1547#page=1",
        "search_input_xpath": '//*[@id="searchKeyword"]',
        "search_btn_xpath": '//*[@id="content"]/form/fieldset/div[2]/button',
        "results_xpath": '//*[@id="boardList"]/tbody/tr'
    },
    {
        "name": "제주",
        "url": "https://www.jeju.go.kr/news/news/law/jeju2.htm",
        "search_input_xpath": '//*[@id="title"]',
        "search_btn_xpath": '//*[@id="app"]/form/div/div[2]/button',
        "results_xpath": '//*[@id="gosiBody"]/tr'
    },
    {
        "name": "부산",
        "url": "https://www.busan.go.kr/nbgosi",
        "search_input_xpath": '//*[@id="srchText"]',
        "search_btn_xpath": '//*[@id="searchFrm"]/div/div/div/div/span[2]/span[3]/button/span',
        "results_xpath": '//*[@id="contents"]/div[2]/table/tbody/tr'
    },
    {
        "name": "대전",
        "url": "https://www.daejeon.go.kr/drh/drhGosiList.do?gosigbn=A&menuSeq=1908",
        "search_input_xpath": '//*[@id="title"]',
        "search_btn_xpath": '//*[@id="searchForm"]/table/tbody/tr[4]/td/input[2]',
        "results_xpath": '//*[@id="cont-body"]/table/tbody/tr'
    },
    {
        "name": "강원",
        "url": "https://state.gwd.go.kr/portal/bulletin/notification",
        "search_input_xpath": '//*[@id="srchSelect2"]',
        "search_btn_xpath": '//*[@id="content-bx"]/div[2]/div[2]/form/fieldset/fieldset[2]/input[2]',
        "results_xpath": '//*[@id="content-bx"]/div[3]/table/tbody/tr'
    },
    {
        "name": "대구",
        "url": "https://www.daegu.go.kr/index.do?menu_id=00940170",
        "search_input_xpath": '//*[@id="sidoGosiAPIVO"]/div[1]/div[2]/dl[3]/dd/input',
        "search_btn_xpath": '//*[@id="sidoGosiAPIVO"]/div[1]/div[2]/dl[3]/dd/div/input',
        "results_xpath": '//*[@id="bbsList"]/tbody/tr'
    },
    {
        "name": "인천",
        "url": "https://announce.incheon.go.kr/citynet/jsp/sap/SAPGosiBizProcess.do?command=searchList&flag=gosiGL&svp=Y&sido=ic",
        "search_input_xpath": '//*[@id="conTitle"]',
        "search_btn_xpath": '//*[@id="srchCondition"]/tbody/tr[1]/td[2]/table/tbody/tr/td[2]/a',
        "results_xpath": '/html/body/form/table/tbody/tr[2]/td[2]/table/tbody/tr[5]/td/table/tbody/tr'
    },
    {
        "name": "울산",
        "url": "https://www.ulsan.go.kr/u/rep/transfer/notice/list.ulsan?mId=001004002000000000",
        "search_input_xpath": '//*[@id="srchWord"]',
        "search_btn_xpath": '//*[@id="searchField"]/dl/dd[3]/button',
        "results_xpath": '//*[@id="contents_inner"]/div[3]/table/tbody/tr'
    },
    {
        "name": "경남",
        "url": "https://www.gyeongnam.go.kr/index.gyeong?menuCd=DOM_000000135003009001",
        "search_input_xpath": '//*[@id="conTitle"]',
        "search_btn_xpath": '//*[@id="searchSubmit"]',
        "results_xpath": '//*[@id="subCnt"]/div[3]/table/tbody/tr'
    },
    {
        "name": "경북",
        "url": "https://www.gb.go.kr/Main/page.do?mnu_uid=6789&&BD_CODE=gosi_notice",
        "search_input_xpath": '//*[@id="word"]',
        "search_btn_xpath": '//*[@id="contwrap"]/div/div[3]/form/div/fieldset/input[2]',
        "results_xpath": '//*[@id="contwrap"]/div/div[3]/table/tbody/tr'
    },
    {
        "name": "충남",
        "url": "https://www.chungnam.go.kr/cnportal/province/province/list.do?menuNo=500487",
        "search_input_xpath": '//*[@id="searchWrd"]',
        "search_btn_xpath": '//*[@id="frm"]/div[1]/fieldset/ul/li[2]/div[2]/div[2]/button',
        "results_xpath": '//*[@id="content"]/div/div[2]/table/tbody/tr'
    },
    {
        "name": "충북",
        "url": "https://www.chungbuk.go.kr/www/selectGosiPblancList.do?key=422",
        "search_input_xpath": '//*[@id="searchWrd"]',
        "search_btn_xpath": '//*[@id="frm"]/div[1]/fieldset/ul/li[2]/div[2]/div[2]/button',
        "results_xpath": '//*[@id="content"]/div/div[2]/table/tbody/tr'
    },
    {
        "name": "세종",
        "url": "https://www.sejong.go.kr/prog/publicNotice/kor/sub02_030301/C1_1/list.do",
        "search_input_xpath": '//*[@id="findWrite"]',
        "search_btn_xpath": '//*[@id="searchForm"]/fieldset/div/div[3]/div/div[4]/span/input',
        "results_xpath": '//*[@id="txt"]/div[2]/table/tbody/tr'
    }
]

# 3. 검색 및 데이터 추출 함수
# ==============================================================================
def scrape_and_find_links(gov_info, search_term, num_results, driver):
    """
    지자체별 URL에 접속하여 검색을 수행하고, 게시글 제목과 URL 링크를 추출합니다.

    Args:
        gov_info (dict): 지자체 이름, URL, XPath 정보
        search_term (str): 검색어
        num_results (int): 추출할 결과 개수
        driver (webdriver): Selenium 웹드라이버 인스턴스

    Returns:
        list: [{'title': '...', 'url': '...'}, ...] 형태의 게시글 목록
    """
    results = []

    try:
        driver.get(gov_info["url"])
        print(f"✅ {gov_info['name']} 페이지 접속: {gov_info['url']}")

        # 검색 입력 필드 찾기
        search_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, gov_info["search_input_xpath"]))
        )
        search_input.send_keys(search_term)

        # 검색 버튼 클릭 (Enter 키 또는 버튼 클릭)
        try:
            search_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, gov_info["search_btn_xpath"]))
            )
            search_button.click()
        except TimeoutException:
            search_input.send_keys(Keys.ENTER)

        print(f"🔍 '{search_term}' 검색 완료.")

        time.sleep(2)  # 검색 결과 로딩을 위한 대기

        # 게시글 목록 찾기
        result_elements = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, gov_info["results_xpath"]))
        )

        for i, element in enumerate(result_elements[:num_results]):
            try:
                # 게시글 제목 추출
                title_element = element.find_element(By.TAG_NAME, 'a')
                title = title_element.text

                # 게시글 URL 추출
                link = title_element.get_attribute('href')

                # 결과 리스트에 추가
                results.append({"title": title, "url": link})

            except NoSuchElementException:
                # 'a' 태그를 찾지 못할 경우, 다음 요소로 이동
                continue

        print(f"📊 {len(results)}개의 게시글 링크 추출 완료.")

    except (NoSuchElementException, TimeoutException) as e:
        print(f"⚠️ {gov_info['name']} 크롤링 실패: 페이지 요소 찾기 실패 ({e})")
    except Exception as e:
        print(f"❌ {gov_info['name']} 크롤링 중 오류 발생: {e}")

    return results

# 4. 모든 지자체 크롤링 실행
# ==============================================================================
all_results = []
search_term = '평가위원'
num_results = 1

for gov_info in local_governments:
    print("\n" + "="*50)
    print(f"▶️ {gov_info['name']} 크롤링 시작...")
    scraped_data = scrape_and_find_links(gov_info, search_term, num_results, driver)

    # 지자체 이름 정보를 각 결과에 추가
    for data in scraped_data:
        data['local_government'] = gov_info['name']

    all_results.extend(scraped_data)

driver.quit()
print("\n" + "="*50)
print("✅ 모든 크롤링 작업 완료.")


▶️ 전남 크롤링 시작...
✅ 전남 페이지 접속: https://www.jeonnam.go.kr/J0203/boardList.do?menuId=jeonnam0203000000
🔍 '평가위원' 검색 완료.
📊 1개의 게시글 링크 추출 완료.

▶️ 전북 크롤링 시작...
✅ 전북 페이지 접속: https://www.jeonbuk.go.kr/board/list.jeonbuk?boardId=BBS_0000129&menuCd=DOM_000000102002005000&contentsSid=1379&cpath=
🔍 '평가위원' 검색 완료.
📊 1개의 게시글 링크 추출 완료.

▶️ 전북타기관 크롤링 시작...
✅ 전북타기관 페이지 접속: https://www.jeonbuk.go.kr/board/list.jeonbuk?boardId=BBS_0000006&menuCd=DOM_000000102002006000&contentsSid=1380&cpath=
🔍 '평가위원' 검색 완료.
📊 1개의 게시글 링크 추출 완료.

▶️ 광주 크롤링 시작...
✅ 광주 페이지 접속: https://search.gwangju.go.kr/search/front/Search.jsp
❌ 광주 크롤링 중 오류 발생: Message: element click intercepted: Element <a href="#" class="btn_detail">...</a> is not clickable at point (320, 60). Other element would receive the click: <a href="javascript:get_re_search2('평가위원');">...</a>
  (Session info: chrome=140.0.7339.186); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickint

KeyboardInterrupt: 

In [ ]:
!pip install xlsxwriter
import openpyxl

In [ ]:
# 5. DataFrame 생성 및 엑셀 파일 저장
# ==============================================================================
df = pd.DataFrame(all_results, columns=['local_government', 'title', 'url'])

excel_file = "17jijache.xlsx"

try:
    with pd.ExcelWriter(excel_file, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='ji', index=False)
        workbook = writer.book
        worksheet = writer.sheets['ji']

        # URL 열에 하이퍼링크 형식 적용
        # URL 열의 인덱스는 2 (0:지자체, 1:제목, 2:URL)
        for i, url in enumerate(df['url']):
            if pd.isna(url):
                continue
            # Pandas의 to_excel은 1부터 시작하고 헤더를 포함하므로 i+1+1
            worksheet.write_url(i + 1, 2, url, string=url)

        print(f"✅ 데이터가 '{excel_file}' 파일로 성공적으로 저장되었습니다.")

except Exception as e:
    print(f"❌ 엑셀 파일 저장 중 오류 발생: {e}")

# 최종 결과 DataFrame 출력 (디버깅용)
print("\n" + "="*50)
print("📌 최종 DataFrame 미리보기:")
print(df.head())